# Messages API

### Install dependencies
- ogx_client
- anthropic

In [ ]:
# Install 
!pip install ogx_client anthropic

In [22]:
from anthropic import Anthropic
from ogx_client import OgxClient
import requests
import rich

# Configuration
OGX_CONNECTION_URL = "http://ogxserver-service.llama.svc.cluster.local:8321"

# Initialize the Anthropic client
anthropic_client = Anthropic(
    base_url=OGX_CONNECTION_URL,
    api_key="fake",
)

# Initialize the OGX client
ogx_client = OgxClient(
    base_url=OGX_CONNECTION_URL,
    api_key="fake",
)

In [23]:
# List available models
models = ogx_client.models.list()
rich.print(models)

ListModelsResponse(
    data=[
        Model(
            id='vllm-inference/llama-32-3b-instruct',
            created=1781501222,
            owned_by='ogx',
            custom_metadata={
                'model_type': 'llm',
                'provider_id': 'vllm-inference',
                'provider_resource_id': 'llama-32-3b-instruct'
            },
            object='model'
        )
    ],
    object='list'
)

In [24]:
# Extract LLM model details
llm_model = next(
    m for m in models.data
    if m.custom_metadata.get("model_type") == "llm"
)

model_id = llm_model.id
print(f"LLM Model: {model_id}")

LLM Model: vllm-inference/llama-32-3b-instruct


In [25]:
methods = [m for m in dir(anthropic_client.messages) if not m.startswith("_")]
print(sorted(methods))

['batches', 'count_tokens', 'create', 'parse', 'stream', 'with_raw_response', 'with_streaming_response']


In [26]:
message = anthropic_client.messages.create(
    model=model_id,
    max_tokens=80,
    messages=[
        {"role": "user", "content": "What is OGX?"}
    ],
)

print(message.content[0].text)

OGX is a popular brand of hair care products, particularly shampoos and conditioners, that is owned by The Procter & Gamble Company (P&G). OGX stands for "Own Greatness," which reflects the brand's mission to help consumers achieve great hair and great lives.

OGX products are known for their natural ingredients, affordability, and effectiveness. They offer a range of


In [27]:


url = f"{OGX_CONNECTION_URL}/v1/messages/count_tokens"

payload = {
    "model": model_id,
    "messages": [
        {
            "role": "user",
            "content": "What is OGX?"
        }
    ]
}

r = requests.post(url, json=payload)

print(r.status_code)
print(r.text)

200
{"input_tokens":40}


In [29]:
result = anthropic_client.messages.count_tokens(
    model=model_id,
    messages=[
        {"role": "user", "content": "What is OGX?"}
    ],
)
print(result)

MessageTokensCount(input_tokens=40)


In [30]:
for text in [
    "Hello",
    "What is OGX?",
    "Tell me about machine learning.",
    "x " * 100,
]:
    result = anthropic_client.messages.count_tokens(
        model=model_id,
        messages=[
            {"role": "user", "content": text}
        ]
    )
    print(len(text), result.input_tokens)

5 36
12 40
31 41
200 135


In [31]:
# Create a batch
batch = anthropic_client.messages.batches.create(
    requests=[
        {
            "custom_id": "ogx-test-1",
            "params": {
                "model": model_id,
                "max_tokens": 80,
                "messages": [
                    {
                        "role": "user",
                        "content": "What is OGX?"
                    }
                ],
            },
        },
        {
            "custom_id": "ogx-test-2",
            "params": {
                "model": model_id,
                "max_tokens": 80,
                "messages": [
                    {
                        "role": "user",
                        "content": "Explain Kubernetes in one sentence."
                    }
                ],
            },
        },
    ]
)

rich.print(batch)

MessageBatch(
    id='msgbatch_000122b0e1144882911bded0',
    archived_at=None,
    cancel_initiated_at=None,
    created_at=datetime.datetime(2026, 6, 15, 5, 27, 38, 959768, tzinfo=datetime.timezone.utc),
    ended_at=None,
    expires_at=datetime.datetime(2026, 6, 16, 5, 27, 38, 959768, tzinfo=datetime.timezone.utc),
    processing_status='in_progress',
    request_counts=MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=2, succeeded=0),
    results_url=None,
    type='message_batch'
)

In [32]:
#print(anthropic_client.messages.batches.list())
url = f"{OGX_CONNECTION_URL}/v1/messages/batches"

r = requests.get(
    url,
    headers={
        "Accept": "application/json",
    },
)

print(r.status_code)
print(r.text)

200
{"data":[{"id":"msgbatch_000122b0e1144882911bded0","type":"message_batch","processing_status":"in_progress","request_counts":{"processing":2,"succeeded":0,"errored":0,"canceled":0,"expired":0},"created_at":"2026-06-15T05:27:38.959768+00:00","expires_at":"2026-06-16T05:27:38.959768+00:00"}],"has_more":false,"first_id":"msgbatch_000122b0e1144882911bded0","last_id":"msgbatch_000122b0e1144882911bded0"}


In [33]:
try:
    batches = anthropic_client.messages.batches.list()
    print(batches)
except Exception as e:
    print(type(e))
    print(e)

SyncPage[MessageBatch](data=[MessageBatch(id='msgbatch_000122b0e1144882911bded0', archived_at=None, cancel_initiated_at=None, created_at=datetime.datetime(2026, 6, 15, 5, 27, 38, 959768, tzinfo=datetime.timezone.utc), ended_at=None, expires_at=datetime.datetime(2026, 6, 16, 5, 27, 38, 959768, tzinfo=datetime.timezone.utc), processing_status='in_progress', request_counts=MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=2, succeeded=0), results_url=None, type='message_batch')], has_more=False, first_id='msgbatch_000122b0e1144882911bded0', last_id='msgbatch_000122b0e1144882911bded0')


In [34]:
retrieve_batch=anthropic_client.messages.batches.retrieve(batch.id)
rich.print(retrieve_batch)

MessageBatch(
    id='msgbatch_000122b0e1144882911bded0',
    archived_at=None,
    cancel_initiated_at=None,
    created_at=datetime.datetime(2026, 6, 15, 5, 27, 38, 959768, tzinfo=datetime.timezone.utc),
    ended_at=None,
    expires_at=datetime.datetime(2026, 6, 16, 5, 27, 38, 959768, tzinfo=datetime.timezone.utc),
    processing_status='in_progress',
    request_counts=MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=2, succeeded=0),
    results_url=None,
    type='message_batch'
)

In [36]:
results = anthropic_client.messages.batches.results(batch.id)

for result in results:
    rich.print(result)

MessageBatchIndividualResponse(
    custom_id='ogx-test-2',
    result=MessageBatchSucceededResult(
        message=Message(
            id='chatcmpl-bedf5c254855c7dc',
            container=None,
            content=[
                TextBlock(
                    citations=None,
                    text='Kubernetes is an open-source container orchestration system that automates the 
deployment, scaling, and management of containerized applications across a cluster of machines, ensuring high 
availability, scalability, and reliability.',
                    type='text'
                )
            ],
            model='llama-32-3b-instruct',
            role='assistant',
            stop_details=None,
            stop_reason='end_turn',
            stop_sequence=None,
            type='message',
            usage=Usage(
                cache_creation=None,
                cache_creation_input_tokens=None,
                cache_read_input_tokens=None,
                inference_geo=None,
                input_tokens=42,
                output_tokens=40,
                output_tokens_details=None,
                server_tool_use=None,
                service_tier=None
            )
        ),
        type='succeeded'
    )
)

MessageBatchIndividualResponse(
    custom_id='ogx-test-1',
    result=MessageBatchSucceededResult(
        message=Message(
            id='chatcmpl-8e5dd000b021baa9',
            container=None,
            content=[
                TextBlock(
                    citations=None,
                    text='OGX is a popular brand of hair care products, particularly shampoos and conditioners, 
owned by The Procter & Gamble Company. OGX stands for "Own Greatness eXtra," which reflects the brand\'s mission to
help customers achieve their hair care goals.\n\nOGX products are known for their natural ingredients, 
affordability, and wide range of styles and formulas to suit different hair types',
                    type='text'
                )
            ],
            model='llama-32-3b-instruct',
            role='assistant',
            stop_details=None,
            stop_reason='max_tokens',
            stop_sequence=None,
            type='message',
            usage=Usage(
                cache_creation=None,
                cache_creation_input_tokens=None,
                cache_read_input_tokens=None,
                inference_geo=None,
                input_tokens=40,
                output_tokens=80,
                output_tokens_details=None,
                server_tool_use=None,
                service_tier=None
            )
        ),
        type='succeeded'
    )
)

In [37]:
# Create a batch and cancel it
batch = anthropic_client.messages.batches.create(
    requests=[
        {
            "custom_id": "abc123",
            "params": {
                "model": model_id,
                "max_tokens": 80,
                "messages": [
                    {
                        "role": "user",
                        "content": "What is llama stack?"
                    }
                ],
            },
        },
        {
            "custom_id": "abc456",
            "params": {
                "model": model_id,
                "max_tokens": 80,
                "messages": [
                    {
                        "role": "user",
                        "content": "Explain OCP in one sentence."
                    }
                ],
            },
        },
    ]
)

rich.print(batch)

import time
print("Cancelling the batch")
time.sleep(5)
cancel_batch = anthropic_client.messages.batches.cancel(batch.id)
rich.print(cancel_batch)

MessageBatch(
    id='msgbatch_e1fed09ed5fe4e34a84237de',
    archived_at=None,
    cancel_initiated_at=None,
    created_at=datetime.datetime(2026, 6, 15, 5, 28, 19, 151350, tzinfo=datetime.timezone.utc),
    ended_at=None,
    expires_at=datetime.datetime(2026, 6, 16, 5, 28, 19, 151350, tzinfo=datetime.timezone.utc),
    processing_status='in_progress',
    request_counts=MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=2, succeeded=0),
    results_url=None,
    type='message_batch'
)

Cancelling the batch


MessageBatch(
    id='msgbatch_e1fed09ed5fe4e34a84237de',
    archived_at=None,
    cancel_initiated_at=datetime.datetime(2026, 6, 15, 5, 28, 24, 169468, tzinfo=TzInfo(0)),
    created_at=datetime.datetime(2026, 6, 15, 5, 28, 19, 151350, tzinfo=datetime.timezone.utc),
    ended_at=None,
    expires_at=datetime.datetime(2026, 6, 16, 5, 28, 19, 151350, tzinfo=datetime.timezone.utc),
    processing_status='canceling',
    request_counts=MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=2, succeeded=0),
    results_url=None,
    type='message_batch'
)

In [38]:
results = anthropic_client.messages.batches.results(batch.id)
for result in results:
    rich.print(result)

MessageBatchIndividualResponse(custom_id='abc123', result=MessageBatchCanceledResult(type='canceled'))

MessageBatchIndividualResponse(custom_id='abc456', result=MessageBatchCanceledResult(type='canceled'))